# 02 — Embedding
Mengubah setiap chunk teks menjadi vector angka menggunakan model
`thenlper/gte-large` dari HuggingFace.

**Spec model:**
- Dimensi vector : 1024
- Max token      : 512
- Model size     : 0.67 GB (perlu download pertama kali)

## 0. Cek Library
Pastikan semua library yang dibutuhkan sudah terinstall.

In [1]:
import importlib

required = ["sentence_transformers", "langchain_community", "chromadb"]

for lib in required:
    found = importlib.util.find_spec(lib) is not None
    status = "✓ OK" if found else "✗ MISSING — pip install " + lib
    print(f"{lib:25s} {status}")

sentence_transformers     ✓ OK
langchain_community       ✓ OK
chromadb                  ✓ OK


## 1. Load Chunks dari JSON
Load hasil chunking dari 01_chunking.ipynb
tanpa perlu load ulang PDF.

In [2]:
import json

with open("../data/chunks.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(f"Total chunks loaded : {len(data)}")
print(f"\n--- Contoh chunk pertama ---")
print(f"Index   : {data[0]['index']}")
print(f"Halaman : {data[0]['metadata']['page']}")
print(f"Panjang : {len(data[0]['page_content'])} karakter")
print(f"Konten  : {data[0]['page_content'][:200]}")

Total chunks loaded : 666

--- Contoh chunk pertama ---
Index   : 0
Halaman : 0
Panjang : 177 karakter
Konten  : SYSTEM DESIGN
SPECIFICATION (SDS)
Pharmacy Information System (PhIS)
Special Approval Medicine (SAM)
DOCUMENT DATE : 06/05/2025
DOCUMENT VERSION : 1.0
DOCUMENT ID : PhIS/SDS/SAM


In [3]:
from sentence_transformers import SentenceTransformer

print("Loading model... (pertama kali ~670 MB, sabar ya)")

model = SentenceTransformer("thenlper/gte-large")

print(f"Model loaded!")
print(f"Dimensi vector : {model.get_sentence_embedding_dimension()}")
print(f"Max token      : {model.max_seq_length}")

C:\Users\Dev\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model... (pertama kali ~670 MB, sabar ya)


C:\Users\Dev\AppData\Roaming\Python\Python312\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dev\.cache\huggingface\hub\models--thenlper--gte-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 3279.72it/s]


Model loaded!
Dimensi vector : 1024
Max token      : 512


C:\Users\Dev\AppData\Local\Temp\ipykernel_8980\3694778467.py:8: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Dimensi vector : {model.get_sentence_embedding_dimension()}")


## 3. Test Embedding 1 Chunk
Coba embed 1 chunk dulu sebelum proses semua 666 chunks.

In [ ]:
# Test embed 1 chunk dulu
sample_text = data[0]['page_content']

vector = model.encode(sample_text)

print(f"Teks input     : {sample_text[:100]}")
print(f"Panjang teks   : {len(sample_text)} karakter")
print(f"Dimensi vector : {len(vector)}")
print(f"Tipe data      : {type(vector)}")
print(f"\nContoh 5 angka pertama dari vector:")
print(vector[:5])



Teks input     : SYSTEM DESIGN
SPECIFICATION (SDS)
Pharmacy Information System (PhIS)
Special Approval Medicine (SAM)
Panjang teks   : 177 karakter
Dimensi vector : 1024
Tipe data      : <class 'numpy.ndarray'>

Contoh 5 angka pertama dari vector:
[-0.0004103 -0.02382    0.01397    0.00716   -0.0166   ]

Contoh semua angka dari vector:
[-0.0004103 -0.02382    0.01397   ... -0.01813   -0.01347   -0.0171   ]


## 4. Embed Semua Chunks
Proses semua 666 chunks sekaligus menjadi vectors.
Ini akan memakan waktu beberapa menit.

In [6]:
import time

texts = [d['page_content'] for d in data]

print(f"Mulai embedding {len(texts)} chunks...")
start = time.time()

vectors = model.encode(texts, batch_size=32, show_progress_bar=True)

elapsed = time.time() - start

print(f"\nSelesai!")
print(f"Waktu          : {elapsed:.1f} detik")
print(f"Total vectors  : {len(vectors)}")
print(f"Shape          : {vectors.shape}")

Mulai embedding 666 chunks...


Batches: 100%|██████████| 21/21 [3:30:02<00:00, 600.11s/it]  


Selesai!
Waktu          : 12602.3 detik
Total vectors  : 666
Shape          : (666, 1024)


## 5. Simpan Vectors ke Chroma
Simpan dari RAM ke disk supaya permanent.

In [7]:
import chromadb

# Buat persistent client — simpan ke folder vector_db
client = chromadb.PersistentClient(path="../vector_db")

collection = client.get_or_create_collection(
    name="phis_sds",
    metadata={"hnsw:space": "cosine"}
)

# Masukkan semua vectors
collection.add(
    ids=[str(d['index']) for d in data],
    embeddings=vectors.tolist(),
    documents=[d['page_content'] for d in data],
    metadatas=[d['metadata'] for d in data]
)

print(f"Total tersimpan : {collection.count()} vectors")
print(f"Lokasi          : ../vector_db")
print(f"\nAman! Vectors sudah di disk, bisa restart laptop.")

Total tersimpan : 666 vectors
Lokasi          : ../vector_db

Aman! Vectors sudah di disk, bisa restart laptop.
